In [42]:
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
import pickle
from datetime import datetime

In [43]:
data = pd.read_excel("Data/dataset_comp_1222_without_ceg_ogn.xlsx")

## Calculate historical returns in the period 2018-2020:

In [44]:
# # Define the tickers and date range
# tickers = list(data['SYMBOL'].values) # Add more tickers as needed
# start_date = "2015-11-01"
# end_date = "2021-01-01"

# # Download the data
# yahoo_data = yf.download(tickers, start=start_date, end=end_date, group_by='ticker', auto_adjust=False, progress=False)

# # Extract adjusted close prices
# adj_close_bt = pd.DataFrame()

# # If only one ticker, yfinance doesn't use a multi-indexed DataFrame, handle accordingly
# if len(tickers) == 1:
#     adj_close_bt[tickers[0]] = yahoo_data['Adj Close']
# else:
#     for ticker in tickers:
#         try:
#             adj_close_bt[ticker] = yahoo_data[ticker]['Adj Close']
#         except KeyError:
#             print(f"No data found for {ticker}.")

# # Display the result
# print(adj_close_bt.head())

12 Failed downloads:
['SBNY', 'CEG', 'OGN']: YFPricesMissingError('possibly delisted; no price data found  (1d 2015-11-01 -> 2021-01-01) (Yahoo error = "Data doesn\'t exist for startDate = 1446350400, endDate = 1609477200")')
['MRO', 'CTLT', 'PXD', 'WRK', 'DISH', 'ATVI', 'SIVBQ']: YFTzMissingError('possibly delisted; no timezone found')
['DFS']: YFPricesMissingError('possibly delisted; no price data found  (1d 2015-11-01 -> 2021-01-01) (Yahoo error = "No data found, symbol may be delisted")')
['BF.B']: YFPricesMissingError('possibly delisted; no price data found  (1d 2015-11-01 -> 2021-01-01)')

In [45]:
# #print(data.loc[data['SYMBOL'] == 'BF.B'])
# # Define the tickers and date range
# tickers = ['BF-B'] # Add more tickers as needed
# start_date = "2015-11-01"
# end_date = "2021-01-01"

# # Download the data
# data_bfb = yf.download(tickers, start=start_date, end=end_date, group_by='ticker', auto_adjust=False, progress=False)

# # Extract adjusted close prices
# adj_close_bfb = pd.DataFrame()

# # If only one ticker, yfinance doesn't use a multi-indexed DataFrame, handle accordingly
# if len(tickers) == 1:
#     adj_close_bfb[tickers[0]] = data_bfb['BF-B']['Adj Close']
# else:
#     for ticker in tickers:
#         try:
#             adj_close_bfb[ticker] = data_bfb[ticker]['Adj Close']
#         except KeyError:
#             print(f"No data found for {ticker}.")

# # Display the result
# #print(adj_close_bfb.head())
# adj_close_bt['BF.B'] = adj_close_bfb.values

In [46]:
# adj_close_bt.to_excel("Data/adj_price_yahoo_comp_1222_historical.xlsx")

In [47]:
adj_close_bt = pd.read_excel("Data/adj_price_yahoo_comp_1222_historical.xlsx")
adj_close_bt.index = adj_close_bt.Date
adj_close_bt.drop(columns='Date', inplace=True)


'CEG', 'OGN'

['DFS']: YFPricesMissingError('possibly delisted; no price data found  (1d 2015-11-01 -> 2021-01-01) (Yahoo error = "No data found, symbol may be delisted")')

     

In [48]:
price_historical = pd.read_excel('Data/price_div_comp_1222_historical.xlsm', sheet_name='CLOSE PRICE', header = 4)

price_historical.columns = price_historical.columns.str.replace(r'\(.*?\)', '', regex=True).str.strip()

price_historical = price_historical.iloc[217:1566]

div_rate = pd.read_excel('Data/price_div_comp_1222_historical.xlsm', sheet_name='DIV RATE', header = 4)
div_rate.columns = div_rate.columns.str.replace(r'\(.*?\)', '', regex=True).str.strip()
div_rate = div_rate.iloc[217:1566]

div_date_historical = pd.read_excel('Data/price_div_comp_1222_historical.xlsm', sheet_name='DIV DATE', header = 4)
div_date_historical.columns = div_date_historical.columns.str.replace(r'\(.*?\)', '', regex=True).str.strip()
div_date_historical = div_date_historical.iloc[217:1566]

div_date_historical.columns = price_historical.columns
div_rate.columns = price_historical.columns

price_historical.index = price_historical.Code
div_rate.index = div_rate.Code
div_date_historical.index = div_date_historical.Code

price_historical = price_historical.iloc[:, 1:]
div_rate = div_rate.iloc[:, 1:]
div_date_historical = div_date_historical.iloc[:, 1:]

# --- Step 1: Calculate adjustment factors
adj_factors = pd.DataFrame(1.0, index=price_historical.index, columns=price_historical.columns)

for company in price_historical.columns:
    for i in range(1, len(price_historical)):
        date = price_historical.index[i]
        prev_date = price_historical.index[i - 1]

        # If ex-dividend happens on this day
        if pd.notna(div_date_historical.at[date, company]):
            div = div_rate.at[date, company]
            price_prev = price_historical.at[prev_date, company]
            if price_prev and price_prev != 0:
                factor = (price_prev - div) / price_prev
                adj_factors.at[date, company] = factor

# --- Step 2: Calculate cumulative adjustment factors in reverse (like Yahoo)
cum_factors = adj_factors.iloc[::-1].cumprod().iloc[::-1]

# --- Step 3: Build adjusted prices
adjusted_prices_calculated = price_historical * cum_factors

#print(data.loc[data['SYMBOL'] == 'WRK', 'NAME'])
#print(div_rate['96699P'].unique())
#print(price_historical['96699P'])
#print(adjusted_prices_calculated['96699P'])
adj_close_bt['WRK'] = adjusted_prices_calculated.loc[adj_close_bt.index, '96699P'].values

#print(data.loc[data['SYMBOL'] == 'CTLT', 'NAME'])
#print(div_rate['8866F3'].unique())
#print(price_historical['8866F3'])
#print(adjusted_prices_calculated['8866F3'])
adj_close_bt['CTLT'] = adjusted_prices_calculated.loc[adj_close_bt.index, ['8866F3']].values

#print(data.loc[data['SYMBOL'] == 'MRO', 'NAME'])
#print(div_rate['544682'].unique())
#print(price_historical['544682'])
#print(adjusted_prices_calculated['544682'])
adj_close_bt['MRO'] = adjusted_prices_calculated.loc[adj_close_bt.index, '544682'].values

#print(data.loc[data['SYMBOL'] == 'PXD', 'NAME'])
#print(div_rate['895705'].unique())
#print(price_historical['895705'])
adj_close_bt['PXD'] = adjusted_prices_calculated.loc[adj_close_bt.index, '895705'].values

#print(data.loc[data['SYMBOL'] == 'SBNY', ['NAME','TYPE']])
#print(div_rate['28709C'].unique())
#print(price_historical['28709C'])
adj_close_bt['SBNY'] = adjusted_prices_calculated.loc[adj_close_bt.index, '28709C'].values

#print(data.loc[data['SYMBOL'] == 'ATVI', ['NAME','TYPE']])
#print(div_rate['312367'].unique())
#print(price_historical['312367'])
adj_close_bt['ATVI'] = adjusted_prices_calculated.loc[adj_close_bt.index, '312367'].values

#print(data.loc[data['SYMBOL'] == 'DISH', ['NAME','TYPE']])
#print(div_rate['135448'].unique())
#print(price_historical['135448'])
adj_close_bt['DISH'] = adjusted_prices_calculated.loc[adj_close_bt.index, '135448'].values

#print(data.loc[data['SYMBOL'] == 'SIVBQ', ['NAME','TYPE']])
#print(div_rate['518628'].unique())
#print(price_historical['518628'])
adj_close_bt['SIVBQ'] = adjusted_prices_calculated.loc[adj_close_bt.index, '518628'].values

print(div_rate['50642F'].unique())
print(price_historical['50642F'])
adj_close_bt['DFS'] = adjusted_prices_calculated.loc[adj_close_bt.index, '50642F'].values

[ nan 0.28 0.3  0.35 0.4  0.44]
Code
2015-11-02    56.28
2015-11-03    56.35
2015-11-04    56.37
2015-11-05    56.91
2015-11-06    57.61
              ...  
2020-12-25    88.29
2020-12-28    88.29
2020-12-29    88.02
2020-12-30    89.30
2020-12-31    90.53
Name: 50642F, Length: 1349, dtype: float64


In [49]:
# ceg and ogn already excluded

In [50]:
#right now my log returns go from 15 dec 2020 to 15 dec 2021 (so the data goes back from 15 nov 2020)
# I could back test three years before so :

# 15 nov 2017 to 15 nov 2018
# 15 nov 2018 to 15 nov 2019
# 15 nov 2019 to 15 nov 2020

In [51]:

adj_close_bt_above_2018 = adj_close_bt.loc[adj_close_bt.index > datetime(2017, 11, 15)]
cols_with_nans = adj_close_bt_above_2018.columns[adj_close_bt_above_2018.isna().any()]
print("Columns with NaNs:", cols_with_nans.tolist())

nan_columns = adj_close_bt_above_2018.loc[:, adj_close_bt_above_2018.isna().any(axis=0)]

for col in nan_columns.columns:
    first_valid = nan_columns[col].first_valid_index()
    print(f"First non-NaN in column '{col}': {first_valid}") # these stocks had later IPO

Columns with NaNs: ['DAY', 'MRNA', 'DOW', 'FOXA', 'CARR', 'OTIS', 'CTVA', 'OGN', 'CEG', 'VICI']
First non-NaN in column 'DAY': 2018-04-26 00:00:00
First non-NaN in column 'MRNA': 2018-12-07 00:00:00
First non-NaN in column 'DOW': 2019-03-20 00:00:00
First non-NaN in column 'FOXA': 2019-03-12 00:00:00
First non-NaN in column 'CARR': 2020-03-19 00:00:00
First non-NaN in column 'OTIS': 2020-03-19 00:00:00
First non-NaN in column 'CTVA': 2019-05-24 00:00:00
First non-NaN in column 'OGN': None
First non-NaN in column 'CEG': None
First non-NaN in column 'VICI': 2018-01-02 00:00:00


VICI is okay, OGN and CEG I have to remove them in any case. For the rest I can take the optimized portfolio, and exclude these 7 stocks and then I can see how does it change in practice the optimised portfolio. After doing this I can evaluate the portfolio back in time. 

In [52]:

nan_backtest = ['DAY', 'MRNA', 'DOW', 'FOXA', 'CARR', 'OTIS', 'CTVA', 'VICI']
for stock in nan_backtest:
    stock_type_miss = data.loc[data['SYMBOL'] == stock, 'TYPE'].values
    if price_historical[stock_type_miss].first_valid_index() < nan_columns[stock].first_valid_index():
        print(stock, price_historical[stock_type_miss].first_valid_index())

VICI 2017-10-17 00:00:00


In [53]:
vici = price_historical['9174A0']
vici = vici[vici.index >= datetime(2017, 12, 1)]

In [54]:
adj_close_bt = adj_close_bt[(adj_close_bt.index >= datetime(2017, 12, 1))]

In [55]:
adj_close_bt['VICI'] = vici

In [56]:
adj_close_bt = adj_close_bt.drop(columns=['OGN', 'CEG', 'DAY', 'MRNA', 'DOW', 'FOXA', 'CARR', 'OTIS', 'CTVA'])

In [57]:
def calculate_monthly_returns(df):
    # Clean data: replace zeros and drop fully NaN rows
    df = df.replace(0.0, np.nan).dropna(how="all")
    
    # Resample to monthly frequency using last available price in each month
    df_monthly = df.resample("ME").last()
    
    # Calculate monthly simple returns (percentage change)
    df_returns = df_monthly.pct_change().dropna()
    
    # Optional: set index to mid-month for visualization consistency
    df_returns.index = df_returns.index.map(lambda x: x.replace(day=15))
    
    return df_returns

In [58]:
monthly_returns_2018_2020 = calculate_monthly_returns(adj_close_bt)

In [59]:
daily_returns_2018_2020 = adj_close_bt.pct_change().dropna()

## Realized TEs (with 2% TE Constraint) in the period 2018-2020:

In [60]:

with open('Data/optimal_portfolios_1222_without_ceg_ogn_minimum_weight.pkl', 'rb') as f:
    optimal_portfolios_shrink = pickle.load(f)


In [61]:
def compute_buy_and_hold_returns(returns_df, weights, stock_labels):
    """
    Compute buy-and-hold (non-rebalanced) portfolio returns.
    
    Args:
        returns_df: DataFrame with stock returns (monthly), index = date, columns = tickers
        weights: np.array of initial weights (must match stock_labels)
        stock_labels: list or Index of tickers in this sector

    Returns:
        Series of portfolio returns over time (monthly)
    """
    # Subset return data for the sector
    sector_returns = returns_df[stock_labels]
    
    # Compute cumulative returns per stock
    cumulative_returns = (1 + sector_returns).cumprod()
    
    # Multiply by initial weights
    weighted_growth = cumulative_returns.multiply(weights, axis=1)
    
    # Total portfolio value each month
    portfolio_value = weighted_growth.sum(axis=1)
    
    # Convert to monthly portfolio returns
    portfolio_returns = portfolio_value.pct_change().dropna()

    return portfolio_returns

def compute_fixed_weight_returns(returns_df, weights, stock_labels):
    """
    Compute fixed-weight portfolio returns (decarbonised portfolio).
    
    Args:
        returns_df: DataFrame of monthly returns
        weights: np.array of optimised weights
        stock_labels: list or Index of stock tickers

    Returns:
        Series of portfolio returns over time
    """
    sector_returns = returns_df[stock_labels]
    portfolio_returns = sector_returns.dot(weights)
    return portfolio_returns

In [62]:
exclude_list = ['DAY', 'MRNA', 'DOW', 'FOXA', 'CARR', 'OTIS', 'CTVA']

# Copy to avoid modifying in place
adjusted_portfolios = {}

for sector, data in optimal_portfolios_shrink.items():
    stock_labels = data['stock_labels']
    w_b = data['w_b_vec']
    w_opt = data['w_opt']
    
    # Create a mask for stocks not in the exclude list
    keep_mask = ~stock_labels.isin(exclude_list)
    
    # Apply mask to weights and stock_labels
    w_b_new = w_b[keep_mask]
    w_opt_new = w_opt[keep_mask]
    labels_new = stock_labels[keep_mask]
    
    # Re-normalize weights to sum to 1
    w_b_new /= w_b_new.sum()
    w_opt_new /= w_opt_new.sum()
    
    # Store in new dictionary
    adjusted_portfolios[sector] = {
        'w_b_vec': w_b_new,
        'w_opt': w_opt_new,
        'stock_labels': labels_new
    }

    for sector, data in adjusted_portfolios.items():
        assert f"{data['w_b_vec'].sum():.4f}" == "1.0000"
        assert f"{data['w_opt'].sum():.4f}"  == "1.0000"

buy_and_hold_sector_returns = {}

for sector, data in adjusted_portfolios.items():
    w_b = data['w_b_vec']                 # initial benchmark weights
    tickers = data['stock_labels']       # stock labels for the sector
    
    # Compute sector-level buy-and-hold benchmark returns
    sector_returns = compute_buy_and_hold_returns(
        monthly_returns_2018_2020,
        weights=w_b,
        stock_labels=tickers
    )
    
    buy_and_hold_sector_returns[sector] = sector_returns

buy_and_hold_df = pd.DataFrame(buy_and_hold_sector_returns)



decarb_sector_returns = {}

for sector, data in adjusted_portfolios.items():
    w_opt = data['w_opt']
    tickers = data['stock_labels']
    
    sector_returns = compute_fixed_weight_returns(
        monthly_returns_2018_2020,
        weights=w_opt,
        stock_labels=tickers
    )
    
    decarb_sector_returns[sector] = sector_returns


sector_tracking_errors = {}

for sector in buy_and_hold_sector_returns:
    # Get returns
    r_b = buy_and_hold_sector_returns[sector]
    r_d = decarb_sector_returns[sector]
    
    # Align indices (just in case)
    common_index = r_b.index.intersection(r_d.index)
    active_returns = r_d.loc[common_index] - r_b.loc[common_index]
    
    # Monthly tracking error
    te_monthly = active_returns.std()
    
    # Annualised tracking error
    te_annualised = te_monthly * np.sqrt(12)
    
    # Store
    sector_tracking_errors[sector] = te_annualised


te_df_total = pd.DataFrame.from_dict(sector_tracking_errors, orient='index', columns=['Annualised Tracking Error'])
te_df_total = te_df_total.sort_values(by='Annualised Tracking Error', ascending=False)
print(te_df_total)

                        Annualised Tracking Error
Consumer Discretionary                   0.080405
Real Estate                              0.042253
Financials                               0.041649
Materials                                0.039573
Health Care                              0.039284
Industrials                              0.037729
Energy                                   0.034594
Utilities                                0.030863
Information Technology                   0.028966
Consumer Staples                         0.026528
Communication Services                   0.014441


In [63]:
def expected_shortfall(x: pd.Series, alpha: float = 0.05) -> float:
    """
    ES/CVaR of a return series at level alpha.
    Returns the (negative) mean of the worst alpha tail (i.e., average loss).
    If the series is too short, returns np.nan.
    """
    x = pd.Series(x).dropna()
    if len(x) == 0:
        return np.nan
    var_alpha = np.percentile(x, 100 * alpha)
    tail = x[x <= var_alpha]
    if len(tail) == 0:
        return np.nan
    return tail.mean()  # likely negative for downside

sector_metrics = {}

for sector in buy_and_hold_sector_returns:
    r_b = buy_and_hold_sector_returns[sector]
    r_d = decarb_sector_returns[sector]

    # align
    idx = r_b.index.intersection(r_d.index)
    active = (r_d.loc[idx] - r_b.loc[idx]).dropna()

    if len(active) < 6:
        continue

    # Standard (variance-based) TE
    te_monthly = active.std()
    te_annual = te_monthly * np.sqrt(12)

    # Tail TE (ES / CVaR). Report absolute magnitude for readability.
    es_5 = expected_shortfall(active, alpha=0.05)         # negative number (downside)
    es_1 = expected_shortfall(active, alpha=0.01)         # deeper tail
    es_5_abs = abs(es_5)
    es_1_abs = abs(es_1)

    sector_metrics[sector] = {
        "TE_Annual": te_annual,          # variance-based, annualised
        "ES_5pct_monthly": es_5,         # negative value (avg worst 5% months)
        "ES_1pct_monthly": es_1,         # negative value (avg worst 1% months)
        "ES_5pct_abs": es_5_abs,         # positive magnitude
        "ES_1pct_abs": es_1_abs,
    }

te_df = pd.DataFrame.from_dict(sector_metrics, orient='index')
# Optional formatting in bps / %
print(te_df.sort_values("TE_Annual", ascending=False))

                        TE_Annual  ES_5pct_monthly  ES_1pct_monthly  \
Consumer Discretionary   0.080405        -0.071014        -0.086909   
Real Estate              0.042253        -0.032176        -0.042913   
Financials               0.041649        -0.028180        -0.033468   
Materials                0.039573        -0.020735        -0.024863   
Health Care              0.039284        -0.016210        -0.016229   
Industrials              0.037729        -0.027998        -0.031105   
Energy                   0.034594        -0.014934        -0.016176   
Utilities                0.030863        -0.014804        -0.016897   
Information Technology   0.028966        -0.016427        -0.018453   
Consumer Staples         0.026528        -0.011876        -0.013271   
Communication Services   0.014441        -0.009353        -0.011224   

                        ES_5pct_abs  ES_1pct_abs  
Consumer Discretionary     0.071014     0.086909  
Real Estate                0.032176     0.042

In [64]:

sector_metrics = {}  # New dictionary to store all metrics

rf_monthly = 0.0  # Assumed risk-free rate (adjust if needed)

for sector in buy_and_hold_sector_returns:
    # Get returns
    r_b = buy_and_hold_sector_returns[sector]
    r_d = decarb_sector_returns[sector]
    
    # Align indices
    common_index = r_b.index.intersection(r_d.index)
    r_b = r_b.loc[common_index]
    r_d = r_d.loc[common_index]
    active_returns = r_d - r_b

    # === Tracking Error ===
    te_monthly = active_returns.std()
    te_annualised = te_monthly * np.sqrt(12)

    # === Benchmark metrics ===
    ret_b = r_b.mean() * 12
    vol_b = r_b.std() * np.sqrt(12)
    sharpe_b = (ret_b - rf_monthly * 12) / vol_b if vol_b != 0 else np.nan

    # === Decarbonised metrics ===
    ret_d = r_d.mean() * 12
    vol_d = r_d.std() * np.sqrt(12)
    sharpe_d = (ret_d - rf_monthly * 12) / vol_d if vol_d != 0 else np.nan

    # Store all metrics
    sector_metrics[sector] = {
        "Annualised Tracking Error": round(te_annualised, 3),
        "Return_B": round(ret_b, 2),
        "Vol_B": round(vol_b, 2),
        "Sharpe_B": round(sharpe_b, 2),
        "Return_D": round(ret_d, 2),
        "Vol_D": round(vol_d, 2),
        "Sharpe_D": round(sharpe_d, 2)
    }

# Convert to DataFrame
metrics_df = pd.DataFrame.from_dict(sector_metrics, orient='index')
metrics_df = metrics_df.sort_values(by="Annualised Tracking Error", ascending=False)
print(metrics_df)


                        Annualised Tracking Error  Return_B  Vol_B  Sharpe_B  \
Consumer Discretionary                      0.080      0.39   0.30      1.29   
Real Estate                                 0.042      0.13   0.16      0.83   
Financials                                  0.042      0.12   0.22      0.58   
Materials                                   0.040      0.14   0.20      0.71   
Health Care                                 0.039      0.16   0.16      1.00   
Industrials                                 0.038      0.13   0.23      0.59   
Energy                                      0.035     -0.08   0.39     -0.22   
Utilities                                   0.031      0.14   0.14      1.00   
Information Technology                      0.029      0.38   0.24      1.62   
Consumer Staples                            0.027      0.14   0.14      0.97   
Communication Services                      0.014      0.16   0.19      0.81   

                        Return_D  Vol_D

Communication Services is the only one with higher Sharpe Ratio

## Testing TEs across 6 months time windows (period 2018-2020), with 2% TE constraint:

In [65]:

sector_tracking_errors = {}
sector_tracking_errors_6m = {}

for sector in buy_and_hold_sector_returns:
    r_b = buy_and_hold_sector_returns[sector]
    r_d = decarb_sector_returns[sector]

    # Align indices
    common_index = r_b.index.intersection(r_d.index)
    r_b = r_b.loc[common_index]
    r_d = r_d.loc[common_index]

    df = pd.DataFrame({'r_b': r_b, 'r_d': r_d})
    df['active_return'] = df['r_d'] - df['r_b']

    six_month_windows = df.resample('6ME')
    te_annualised_list = []

    for _, window in six_month_windows:
        if len(window) >= 3:
            te_6m = window['active_return'].std()
            te_annualised = te_6m * np.sqrt(2)
            te_annualised_list.append(te_annualised)

    sector_tracking_errors_6m[sector] = te_annualised_list
    sector_tracking_errors[sector] = (
        np.mean(te_annualised_list) if te_annualised_list else np.nan
    )

# --- Build Table ---
# Get the max number of 6M periods across all sectors (to align column count)
max_periods = max(len(v) for v in sector_tracking_errors_6m.values())

# Build rows for DataFrame
rows = {}
for sector, te_list in sector_tracking_errors_6m.items():
    row = te_list + [np.nan] * (max_periods - len(te_list))  # pad with NaNs
    row.append(np.mean(te_list) if te_list else np.nan)
    rows[sector] = row

# Create column labels
columns = [f"TE_time_window_{i+1}" for i in range(max_periods)] + ["Avg_TE"]

# Create and display DataFrame
te_df = pd.DataFrame.from_dict(rows, orient='index', columns=columns)
print(te_df.round(4).sort_values(by='Avg_TE', ascending=False))


                        TE_time_window_1  TE_time_window_2  TE_time_window_3  \
Consumer Discretionary            0.0093            0.0116            0.0111   
Financials                        0.0106            0.0138            0.0157   
Health Care                       0.0102            0.0224            0.0107   
Real Estate                       0.0080            0.0105            0.0062   
Industrials                       0.0083            0.0146            0.0094   
Materials                         0.0080            0.0110            0.0105   
Energy                            0.0046            0.0053            0.0156   
Utilities                         0.0037            0.0121            0.0078   
Information Technology            0.0143            0.0095            0.0110   
Consumer Staples                  0.0055            0.0197            0.0066   
Communication Services            0.0043            0.0103            0.0037   

                        TE_time_window_

In [66]:
import numpy as np
import pandas as pd

# Containers
sector_tracking_errors_6m = {}
sharpe_ratios_b_6m = {}
sharpe_ratios_d_6m = {}
returns_b_6m = {}
returns_d_6m = {}
vol_b_6m = {}
vol_d_6m = {}

summary_metrics = {}

for sector in buy_and_hold_sector_returns:
    r_b = buy_and_hold_sector_returns[sector]
    r_d = decarb_sector_returns[sector]

    # Align indices
    common_index = r_b.index.intersection(r_d.index)
    r_b = r_b.loc[common_index]
    r_d = r_d.loc[common_index]

    df = pd.DataFrame({'r_b': r_b, 'r_d': r_d})
    df['active_return'] = df['r_d'] - df['r_b']

    six_month_windows = df.resample('6ME')

    te_annualised_list = []
    sharpe_b_list = []
    sharpe_d_list = []
    return_b_list = []
    return_d_list = []
    vol_b_list = []
    vol_d_list = []

    for _, window in six_month_windows:
        if len(window) >= 3:
            # --- Tracking Error ---
            te_6m = window['active_return'].std()
            te_annualised = te_6m * np.sqrt(2)
            te_annualised_list.append(te_annualised)

            # --- Benchmark Metrics ---
            r_b_win = window['r_b']
            mean_b = r_b_win.mean() * 12
            std_b = r_b_win.std() * np.sqrt(12)
            sharpe_b = mean_b / std_b if std_b != 0 else np.nan
            return_b_list.append(mean_b)
            vol_b_list.append(std_b)
            sharpe_b_list.append(sharpe_b)

            # --- Decarb Metrics ---
            r_d_win = window['r_d']
            mean_d = r_d_win.mean() * 12
            std_d = r_d_win.std() * np.sqrt(12)
            sharpe_d = mean_d / std_d if std_d != 0 else np.nan
            return_d_list.append(mean_d)
            vol_d_list.append(std_d)
            sharpe_d_list.append(sharpe_d)

    # Store lists per sector
    sector_tracking_errors_6m[sector] = te_annualised_list
    returns_b_6m[sector] = return_b_list
    returns_d_6m[sector] = return_d_list
    vol_b_6m[sector] = vol_b_list
    vol_d_6m[sector] = vol_d_list
    sharpe_ratios_b_6m[sector] = sharpe_b_list
    sharpe_ratios_d_6m[sector] = sharpe_d_list

    # Store averaged metrics in summary table
    summary_metrics[sector] = {
        "Avg_TE": np.nanmean(te_annualised_list),
        "Avg_Return_B": np.nanmean(return_b_list),
        "Avg_Return_D": np.nanmean(return_d_list),
        "Avg_Vol_B": np.nanmean(vol_b_list),
        "Avg_Vol_D": np.nanmean(vol_d_list),
        "Avg_Sharpe_B": np.nanmean(sharpe_b_list),
        "Avg_Sharpe_D": np.nanmean(sharpe_d_list)
    }

# Get max number of 6M periods
max_periods = max(len(v) for v in sector_tracking_errors_6m.values())

# Build detailed rows
detailed_rows = {}

for sector in sector_tracking_errors_6m:
    row = {}

    # TE per 6M
    te_list = sector_tracking_errors_6m[sector]
    for i, val in enumerate(te_list):
        row[f"TE_{i+1}"] = val
    row["Avg_TE"] = np.nanmean(te_list)

    # Returns Benchmark
    ret_b_list = returns_b_6m[sector]
    for i, val in enumerate(ret_b_list):
        row[f"Return_B_{i+1}"] = val
    row["Avg_Return_B"] = np.nanmean(ret_b_list)

    # Returns Decarb
    ret_d_list = returns_d_6m[sector]
    for i, val in enumerate(ret_d_list):
        row[f"Return_D_{i+1}"] = val
    row["Avg_Return_D"] = np.nanmean(ret_d_list)

    # Volatility Benchmark
    vol_b_list = vol_b_6m[sector]
    for i, val in enumerate(vol_b_list):
        row[f"Vol_B_{i+1}"] = val
    row["Avg_Vol_B"] = np.nanmean(vol_b_list)

    # Volatility Decarb
    vol_d_list = vol_d_6m[sector]
    for i, val in enumerate(vol_d_list):
        row[f"Vol_D_{i+1}"] = val
    row["Avg_Vol_D"] = np.nanmean(vol_d_list)

    # Sharpe Benchmark
    sharpe_b_list = sharpe_ratios_b_6m[sector]
    for i, val in enumerate(sharpe_b_list):
        row[f"Sharpe_B_{i+1}"] = val
    row["Avg_Sharpe_B"] = np.nanmean(sharpe_b_list)

    # Sharpe Decarb
    sharpe_d_list = sharpe_ratios_d_6m[sector]
    for i, val in enumerate(sharpe_d_list):
        row[f"Sharpe_D_{i+1}"] = val
    row["Avg_Sharpe_D"] = np.nanmean(sharpe_d_list)

    detailed_rows[sector] = row
# Create full table
detailed_df = pd.DataFrame.from_dict(detailed_rows, orient='index')

# Round columns selectively
rounded_df = detailed_df.copy()
for col in rounded_df.columns:
    if col.startswith("TE_") or col == "Avg_TE":
        rounded_df[col] = rounded_df[col].round(3)
    else:
        rounded_df[col] = rounded_df[col].round(2)

# Optional: sort by average Sharpe_D
rounded_df = rounded_df.sort_values(by="Avg_Sharpe_D", ascending=False)

# Display the full table
pd.set_option('display.max_columns', None)
print(rounded_df)


                         TE_1   TE_2   TE_3   TE_4   TE_5   TE_6  Avg_TE  \
Utilities               0.004  0.012  0.008  0.013  0.022  0.013   0.012   
Real Estate             0.008  0.010  0.006  0.005  0.031  0.028   0.015   
Information Technology  0.014  0.009  0.011  0.015  0.010  0.011   0.012   
Consumer Discretionary  0.009  0.012  0.011  0.023  0.050  0.060   0.027   
Consumer Staples        0.005  0.020  0.007  0.009  0.006  0.014   0.010   
Health Care             0.010  0.022  0.011  0.019  0.016  0.017   0.016   
Communication Services  0.004  0.010  0.004  0.004  0.006  0.002   0.005   
Materials               0.008  0.011  0.011  0.014  0.027  0.011   0.014   
Industrials             0.008  0.015  0.009  0.006  0.027  0.018   0.014   
Financials              0.011  0.014  0.016  0.017  0.023  0.022   0.017   
Energy                  0.005  0.005  0.016  0.012  0.018  0.020   0.013   

                        Return_B_1  Return_B_2  Return_B_3  Return_B_4  \
Utilities    

In [67]:
summary_df = pd.DataFrame.from_dict(summary_metrics, orient='index')

# Optional: round and sort
summary_df = summary_df.round({
    "Avg_TE": 3,
    "Avg_Return_B": 2,
    "Avg_Return_D": 2,
    "Avg_Vol_B": 2,
    "Avg_Vol_D": 2,
    "Avg_Sharpe_B": 2,
    "Avg_Sharpe_D": 2
})

summary_df = summary_df.sort_values(by="Avg_Sharpe_D", ascending=False)

# Display full columns
pd.set_option('display.max_columns', None)
print(summary_df)


                        Avg_TE  Avg_Return_B  Avg_Return_D  Avg_Vol_B  \
Utilities                0.012          0.16          0.16       0.13   
Real Estate              0.015          0.16          0.17       0.14   
Information Technology   0.012          0.39          0.35       0.24   
Consumer Discretionary   0.027          0.42          0.34       0.28   
Consumer Staples         0.010          0.17          0.16       0.14   
Health Care              0.016          0.18          0.19       0.17   
Communication Services   0.005          0.18          0.20       0.19   
Materials                0.014          0.18          0.19       0.19   
Industrials              0.014          0.17          0.16       0.23   
Financials               0.017          0.15          0.16       0.22   
Energy                   0.013         -0.03         -0.00       0.38   

                        Avg_Vol_D  Avg_Sharpe_B  Avg_Sharpe_D  
Utilities                    0.13          2.10          2.

- 5 cases in which returns are higher (and 2 are same)
- 3 cases in which volatility is lower (and 2 are same)
- 4 cases in which sharpe ratio is higher


## Realized TEs during COVID crash (2% TE constaint) -  Regime-specific TE (Stress TE)


For the COVID crash / 2020 backtest:
U use fixed weights for both the benchmark and the decarbonized portfolio.

In [68]:
def compute_fixed_weight_returns(returns_df, weights, stock_labels):
    """
    Compute fixed-weight portfolio returns from daily returns.
    
    Args:
        returns_df: DataFrame of daily returns (index = date, columns = tickers)
        weights: np.array or Series of weights matching stock_labels
        stock_labels: list of stock tickers in the portfolio

    Returns:
        Series of daily portfolio returns
    """
    sector_returns = returns_df[stock_labels]
    weights_series = pd.Series(weights, index=stock_labels)
    portfolio_returns = sector_returns.dot(weights_series)
    return portfolio_returns

decarb_sector_daily_returns = {}

for sector, data in adjusted_portfolios.items():
    w_opt = data['w_opt']
    tickers = data['stock_labels']
    
    sector_returns = compute_fixed_weight_returns(
        daily_returns_2018_2020,
        weights=w_opt,
        stock_labels=tickers
    )
    
    decarb_sector_daily_returns[sector] = sector_returns

buy_and_hold_sector_daily_returns = {}

for sector, data in adjusted_portfolios.items():
    w_b = data['w_b_vec']                 # initial benchmark weights
    tickers = data['stock_labels']       # stock labels for the sector
    
    # Compute sector-level buy-and-hold benchmark returns
    sector_returns = compute_fixed_weight_returns(
        daily_returns_2018_2020,
        weights=w_b,
        stock_labels=tickers
    )
    
    buy_and_hold_sector_daily_returns[sector] = sector_returns


covid_crash_start = "2020-02-19"
covid_crash_end = "2020-03-23"

sector_tracking_errors = {}

for sector in buy_and_hold_sector_daily_returns:
    r_b = buy_and_hold_sector_daily_returns[sector].loc[covid_crash_start:covid_crash_end]
    r_d = decarb_sector_daily_returns[sector].loc[covid_crash_start:covid_crash_end]

    # Skip if not enough data
    if len(r_b) < 5 or len(r_d) < 5:
        continue

    active_returns = r_d - r_b
    te_daily = active_returns.std()
    te_annualised = te_daily * np.sqrt(252)

    sector_tracking_errors[sector] = te_annualised

covid_te = pd.DataFrame.from_dict(sector_tracking_errors, orient='index', columns=["TE_Crash_Annualised"])
print(covid_te.round(4))


                        TE_Crash_Annualised
Consumer Discretionary               0.0758
Health Care                          0.0873
Utilities                            0.0954
Information Technology               0.0567
Real Estate                          0.0708
Materials                            0.0887
Industrials                          0.0827
Financials                           0.0711
Energy                               0.0903
Communication Services               0.0341
Consumer Staples                     0.0572


In [69]:
# Define risk-free rate (e.g., 0 for simplicity during crisis)
rf_daily = 0.0

results = []

for sector in buy_and_hold_sector_daily_returns:
    r_b = buy_and_hold_sector_daily_returns[sector].loc[covid_crash_start:covid_crash_end]
    r_d = decarb_sector_daily_returns[sector].loc[covid_crash_start:covid_crash_end]

    # Skip if not enough data
    if len(r_b) < 5 or len(r_d) < 5:
        continue

    # --- Tracking Error ---
    active_returns = r_d - r_b
    te_daily = active_returns.std()
    te_annualised = te_daily * np.sqrt(252)

    # --- Benchmark stats ---
    mean_b = r_b.mean() * 252
    vol_b = r_b.std() * np.sqrt(252)
    sharpe_b = (mean_b - rf_daily * 252) / vol_b if vol_b != 0 else np.nan

    # --- Decarb stats ---
    mean_d = r_d.mean() * 252
    vol_d = r_d.std() * np.sqrt(252)
    sharpe_d = (mean_d - rf_daily * 252) / vol_d if vol_d != 0 else np.nan

    # Store all results
    results.append({
        "Sector": sector,
        "TE_Crash_Annualised": te_annualised,
        "Return_B": mean_b,
        "Vol_B": vol_b,
        "Sharpe_B": sharpe_b,
        "Return_D": mean_d,
        "Vol_D": vol_d,
        "Sharpe_D": sharpe_d
    })

# Create DataFrame
covid_df = pd.DataFrame(results)

# Round appropriately
covid_df["TE_Crash_Annualised"] = covid_df["TE_Crash_Annualised"].round(3)
covid_df[["Return_B", "Return_D", "Vol_B", "Vol_D", "Sharpe_B", "Sharpe_D"]] = \
    covid_df[["Return_B", "Return_D", "Vol_B", "Vol_D", "Sharpe_B", "Sharpe_D"]].round(2)

# Sort if needed
covid_df = covid_df.sort_values(by="TE_Crash_Annualised", ascending=False)

# Display
print(covid_df)


                    Sector  TE_Crash_Annualised  Return_B  Vol_B  Sharpe_B  \
2                Utilities                0.095     -4.32   0.86     -5.04   
8                   Energy                0.090     -7.52   1.12     -6.72   
5                Materials                0.089     -4.38   0.79     -5.54   
1              Health Care                0.087     -3.13   0.70     -4.49   
6              Industrials                0.083     -4.97   0.78     -6.37   
0   Consumer Discretionary                0.076     -4.19   0.78     -5.36   
4              Real Estate                0.071     -4.65   0.89     -5.20   
7               Financials                0.071     -5.07   0.95     -5.35   
3   Information Technology                0.057     -3.17   0.92     -3.45   
10        Consumer Staples                0.057     -2.56   0.71     -3.63   
9   Communication Services                0.034     -3.20   0.71     -4.50   

    Return_D  Vol_D  Sharpe_D  
2      -4.34   0.82     -5.30  

In most sectors, the decarbonised portfolios had slightly worse Sharpe ratios.

Exceptions:

Materials and Communication Services had slightly better Sharpe under decarbonisation — but only marginally

## Realized TE across differend TE objectives

Test then realized TE across different TE objectives: 

In [70]:
with open("Data/optimal_portfolios_by_te.pkl", "rb") as f:
    optimal_portfolios_by_te = pickle.load(f)


the only one with the higher Sharpe Ratio is communication services:

In [71]:
exclude_list = ['DAY', 'MRNA', 'DOW', 'FOXA', 'CARR', 'OTIS', 'CTVA']

# List of result rows
results = []

for te_cap, sector_data in optimal_portfolios_by_te.items():
    for sector, data in sector_data.items():
        stock_labels = data['stock_labels']
        w_b = data['w_b_vec']
        w_opt = data['w_opt']
        
        # Exclude tickers
        keep_mask = ~stock_labels.isin(exclude_list)
        w_b_new = w_b[keep_mask]
        w_opt_new = w_opt[keep_mask]
        labels_new = stock_labels[keep_mask]

        # Normalize weights
        w_b_new /= w_b_new.sum()
        w_opt_new /= w_opt_new.sum()

        try:
            # Compute returns
            r_b = compute_buy_and_hold_returns(
                monthly_returns_2018_2020,
                weights=w_b_new,
                stock_labels=labels_new
            )

            r_d = compute_fixed_weight_returns(
                monthly_returns_2018_2020,
                weights=w_opt_new,
                stock_labels=labels_new
            )

            # Align indices
            common_index = r_b.index.intersection(r_d.index)
            r_b = r_b.loc[common_index]
            r_d = r_d.loc[common_index]
            active_returns = r_d - r_b

            # TE calculation
            te_monthly = active_returns.std()
            te_annualised = te_monthly * np.sqrt(12)

            # Benchmark stats
            ret_b = r_b.mean() * 12
            vol_b = r_b.std() * np.sqrt(12)
            sharpe_b = ret_b / vol_b if vol_b != 0 else np.nan

            # Optimised stats
            ret_d = r_d.mean() * 12
            vol_d = r_d.std() * np.sqrt(12)
            sharpe_d = ret_d / vol_d if vol_d != 0 else np.nan

            # Store as row
            results.append({
                "Sector": sector,
                "TE_Cap": te_cap,
                "TE_Annualised": te_annualised,
                "Return_B": ret_b,
                "Return_D": ret_d,
                "Vol_B": vol_b,
                "Vol_D": vol_d,
                "Sharpe_B": sharpe_b,
                "Sharpe_D": sharpe_d
            })

        except Exception as e:
            print(f"Failed for {sector} at TE cap {te_cap}: {e}")

df = pd.DataFrame(results)
# Round columns as requested
df["TE_Annualised"] = df["TE_Annualised"].round(3)
df[["Return_B", "Return_D", "Sharpe_B", "Sharpe_D"]] = df[["Return_B", "Return_D", "Sharpe_B", "Sharpe_D"]].round(2)

# Sort and display
df = df.sort_values(by=["TE_Cap", "TE_Annualised"], ascending=[True, False])
print(df)



                    Sector  TE_Cap  TE_Annualised  Return_B  Return_D  \
0   Consumer Discretionary    0.01          0.079      0.39      0.33   
4              Real Estate    0.01          0.038      0.13      0.14   
8                   Energy    0.01          0.034     -0.08     -0.06   
5                Materials    0.01          0.033      0.14      0.15   
7               Financials    0.01          0.030      0.12      0.12   
3   Information Technology    0.01          0.029      0.38      0.35   
6              Industrials    0.01          0.026      0.13      0.13   
1              Health Care    0.01          0.023      0.16      0.16   
2                Utilities    0.01          0.023      0.14      0.13   
10        Consumer Staples    0.01          0.017      0.14      0.14   
9   Communication Services    0.01          0.010      0.16      0.17   
11  Consumer Discretionary    0.02          0.080      0.39      0.32   
15             Real Estate    0.02          0.042  